# Shared code

Settings and helper functions used by all the other notebooks. Each one loads this notebook with
`%run common.ipynb`, which runs every cell below inside that notebook. A change made here
therefore applies everywhere.

Keeping a single copy means the label scheme, clip extraction, resampling and evaluation are the
same for every model, so any difference in results comes from the models themselves.

## 1. Settings

In [1]:
import pathlib

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report, f1_score,
                             roc_auc_score)
from tqdm import tqdm

In [2]:
# Paths
WAVS_DIR = pathlib.Path("wavs")
CLIPS_CSV = "SEP-28k-Extended_clips.csv"
CACHE_DIR = pathlib.Path("feature_cache")
RESULTS_DIR = pathlib.Path("results")
UCLASS_ROOT = pathlib.Path("uclass-stuttered-speech-clips-sep-28k-format/versions/3")

CACHE_DIR.mkdir(exist_ok=True)

# Audio
SAMPLE_RATE = 16000
CLIP_SAMPLES = 3 * SAMPLE_RATE  # every clip is padded/cut to 3 seconds

# Labels. Class index = position in this list, so 0-4 are stutter types and 5 is fluent.
CLASS_NAMES = ["Prolongation", "Block", "SoundRep", "WordRep", "Interjection", "NoStutteredWords"]
LABELS = list(range(len(CLASS_NAMES)))
FLUENT = 5
QUALITY_COLS = ["PoorAudioQuality", "NoSpeech", "Music", "DifficultToUnderstand"]
SPLIT_COL = "SEP28k-E"  # speaker-exclusive train/dev/test split from SEP-28k-Extended
MIN_VOTES = 2           # a label counts if at least 2 of the 3 annotators marked it

SEED = 42

## 2. Loading data

Labels and clips for SEP-28k-E and UCLASS, turning audio into 3-second float arrays and MFCCs, and
caching expensive results in `feature_cache/`.

In [3]:
def load_clips():
    """Load SEP-28k-E clips and give each one a single class label.

    Drops clips that are not in the split, have a quality flag, have no label with
    at least MIN_VOTES votes, or come from an episode we couldn't download.
    The label is the class with the most votes (ties go to the earlier class).
    """
    clips = pd.read_csv(CLIPS_CSV)
    clips["Show"] = clips["Show"].str.strip()
    clips = clips[clips[SPLIT_COL].isin(["train", "dev", "test"])]
    clips = clips[~(clips[QUALITY_COLS] >= 1).any(axis=1)]

    votes = clips[CLASS_NAMES].where(clips[CLASS_NAMES] >= MIN_VOTES, 0)
    clips = clips[(votes > 0).any(axis=1)].copy()
    clips["label"] = np.argmax(votes.loc[clips.index].values, axis=1)

    clips["wav_path"] = [WAVS_DIR / show / f"{ep}.wav" for show, ep in zip(clips["Show"], clips["EpId"])]
    clips = clips[clips["wav_path"].map(pathlib.Path.exists)].reset_index(drop=True)
    return clips

In [4]:
def split_clips(clips):
    """Return the train, dev and test dataframes, sorted so each episode WAV is read once."""
    def part(name):
        df = clips[clips[SPLIT_COL] == name]
        return df.sort_values(["Show", "EpId"]).reset_index(drop=True)
    return part("train"), part("dev"), part("test")

In [5]:
def print_class_counts(labels, title):
    print(title)
    for i, name in enumerate(CLASS_NAMES):
        print(f"  {i} {name}: {(np.asarray(labels) == i).sum()}")

In [6]:
def fix_length(audio, n=CLIP_SAMPLES):
    """Zero-pad or cut audio to exactly n samples."""
    if len(audio) >= n:
        return audio[:n]
    return np.pad(audio, (0, n - len(audio)))

In [7]:
def to_float(audio):
    """int16 PCM -> float32 in [-1, 1]."""
    return audio.astype(np.float32) / 32768.0

In [8]:
def sep28k_clips(df, desc):
    """Yield each clip in df as a 3-second float32 array.

    Clips are cut out of the full episode WAV with the Start/Stop sample indices.
    df is sorted by episode, so we only reopen the WAV when the episode changes.
    """
    current_path, episode = None, None
    for row in tqdm(df.itertuples(), total=len(df), desc=desc):
        if row.wav_path != current_path:
            _, episode = wavfile.read(row.wav_path)
            current_path = row.wav_path
        yield fix_length(to_float(episode[row.Start:row.Stop]))

In [9]:
def load_uclass_meta():
    """UCLASS clip list with labels and age groups (written by UCLASS_Preprocessing.ipynb)."""
    return pd.read_csv(CACHE_DIR / "meta_uclass_v1.csv")

In [10]:
def uclass_clips(meta):
    """Yield each UCLASS clip as a 3-second float32 array (UCLASS clips are already separate files)."""
    for path in tqdm(meta["filepath"], desc="uclass"):
        _, audio = wavfile.read(UCLASS_ROOT / path)
        yield fix_length(to_float(audio))

In [11]:
def mfcc(clip):
    """40 MFCCs with a 25 ms window and 10 ms hop. Returns shape (40, frames)."""
    return librosa.feature.mfcc(y=clip, sr=SAMPLE_RATE, n_mfcc=40, n_fft=400, hop_length=160)

In [12]:
def cached(x_name, y_name, build):
    """Load X and y from feature_cache/ if they exist, otherwise call build() and save the result."""
    x_path = CACHE_DIR / f"{x_name}.npy"
    y_path = CACHE_DIR / f"{y_name}.npy"
    if x_path.exists() and y_path.exists():
        print(f"{x_name}: loaded from cache")
        return np.load(x_path), np.load(y_path)

    X, y = build()
    np.save(x_path, X)
    np.save(y_path, y)
    print(f"{x_name}: {X.shape} saved to cache")
    return X, y

## 3. Class imbalance

The three imbalance strategies compared in every model notebook, and the Random Forest shared by
Models 2 and 3.

In [13]:
def imbalance_strategies(y, seed=SEED):
    """Training-set indices for the three imbalance strategies we compare on dev.

    class_weighted: the full training set (the model's class weights do the balancing)
    undersampling:  NoStutteredWords cut down to the average size of the stutter classes
    oversampling:   every stutter class sampled with replacement up to the NoStutteredWords size
    """
    rng = np.random.default_rng(seed)
    by_class = [np.where(y == c)[0] for c in LABELS]
    stutter, fluent = by_class[:FLUENT], by_class[FLUENT]
    stutter_mean = int(np.mean([len(idx) for idx in stutter]))

    undersampled = np.concatenate(
        stutter + [rng.choice(fluent, size=min(stutter_mean, len(fluent)), replace=False)])
    oversampled = np.concatenate(
        [rng.choice(idx, size=len(fluent), replace=True) for idx in stutter] + [fluent])

    return {
        "class_weighted": np.arange(len(y)),
        "undersampling": undersampled,
        "oversampling": oversampled,
    }

In [14]:
def make_random_forest():
    """The Random Forest used by Models 2 and 3, so the only difference between them is the features."""
    return RandomForestClassifier(n_estimators=100, min_samples_split=5, class_weight="balanced",
                                  n_jobs=-1, random_state=SEED)

## 4. Evaluation

Macro F1 with bootstrap confidence intervals, the standard report for each evaluation set, the
child vs adult breakdown, and saving predictions for `Results_Summary.ipynb`.

In [15]:
def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)

In [16]:
def bootstrap_macro_f1(y_true, y_pred, n_boot=2000, seed=SEED):
    """Macro F1 with a 95% bootstrap confidence interval (clips resampled with replacement)."""
    rng = np.random.default_rng(seed)
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    n = len(y_true)
    scores = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        scores[b] = macro_f1(y_true[idx], y_pred[idx])
    low, high = np.percentile(scores, [2.5, 97.5])
    return macro_f1(y_true, y_pred), low, high

In [17]:
def report(y_true, y_pred, proba, title):
    """Print the classification report and macro AUC, and plot a row-normalised confusion matrix."""
    print(f"=== {title} ===")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    auc = roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
    print(f"Macro AUC-ROC (OVR): {auc:.4f}")

    fig, ax = plt.subplots(figsize=(7, 6))
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=CLASS_NAMES,
                                            normalize="true", ax=ax, colorbar=False)
    ax.set_title(f"Confusion matrix - {title} (row-normalised)")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

In [18]:
def age_group_report(y_true, y_pred, meta):
    """Per-class report and bootstrapped macro F1 for the UCLASS child and adult subgroups."""
    for group in ["Child (<18)", "Adult (18+)"]:
        mask = (meta["age_group"] == group).to_numpy()
        yt, yp = y_true[mask], y_pred[mask]
        print(f"\n=== UCLASS - {group} (n={mask.sum()}) ===")
        print(classification_report(yt, yp, labels=LABELS, target_names=CLASS_NAMES, zero_division=0))
        f1, low, high = bootstrap_macro_f1(yt, yp)
        print(f"Macro F1: {f1:.4f}  (95% bootstrap CI: [{low:.4f}, {high:.4f}])")

In [19]:
def save_predictions(filename, splits, strategy_dev_f1, best_strategy):
    """Save labels, predictions and probabilities for Results_Summary.ipynb.

    splits maps "dev"/"test"/"uclass" to (y_true, y_pred, proba).
    """
    arrays = {}
    for name, (y_true, y_pred, proba) in splits.items():
        arrays[f"{name}_true"] = y_true
        arrays[f"{name}_pred"] = y_pred
        arrays[f"{name}_proba"] = proba

    RESULTS_DIR.mkdir(exist_ok=True)
    np.savez(RESULTS_DIR / filename, **arrays,
             strategy_names=np.array(list(strategy_dev_f1)),
             strategy_dev_f1=np.array(list(strategy_dev_f1.values())),
             best_strategy=np.array(best_strategy))
    print(f"Saved {RESULTS_DIR / filename}")